In [27]:
import pandas as pd

df = pd.read_csv('/lakehouse/default/Files/bronze/bronze_health_indicators.csv', low_memory=False)
print(df.shape)

print(df.dtypes)
print(df.isnull().sum())

StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 34, Finished, Available, Finished, False)

(327165, 18)
REF_DATE             int64
GEO                 object
DGUID               object
Age group           object
Sex                 object
Indicators          object
Characteristics     object
UOM                 object
UOM_ID               int64
SCALAR_FACTOR       object
SCALAR_ID            int64
VECTOR              object
COORDINATE          object
VALUE              float64
STATUS              object
SYMBOL             float64
TERMINATED          object
DECIMALS             int64
dtype: object
REF_DATE                0
GEO                     0
DGUID               26275
Age group               0
Sex                     0
Indicators              0
Characteristics         0
UOM                     0
UOM_ID                  0
SCALAR_FACTOR           0
SCALAR_ID               0
VECTOR                  0
COORDINATE              0
VALUE               78873
STATUS             190842
SYMBOL             327165
TERMINATED         295215
DECIMALS                0
dtype: int64


In [28]:
df = df.drop(columns=[
    'SYMBOL', 'TERMINATED', 'DGUID', 'STATUS', 'SCALAR_FACTOR', 
    'SCALAR_ID', 'VECTOR', 'COORDINATE', 'UOM_ID', 'DECIMALS'])
print(df.shape)

StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 35, Finished, Available, Finished, False)

(327165, 8)


In [29]:
print(df.head())

StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 36, Finished, Available, Finished, False)

   REF_DATE                             GEO                 Age group  \
0      2015  Canada (excluding territories)  Total, 12 years and over   
1      2015  Canada (excluding territories)  Total, 12 years and over   
2      2015  Canada (excluding territories)  Total, 12 years and over   
3      2015  Canada (excluding territories)  Total, 12 years and over   
4      2015  Canada (excluding territories)  Total, 12 years and over   

          Sex                                Indicators  \
0  Both sexes  Perceived health, very good or excellent   
1  Both sexes  Perceived health, very good or excellent   
2  Both sexes  Perceived health, very good or excellent   
3  Both sexes  Perceived health, very good or excellent   
4  Both sexes  Perceived health, very good or excellent   

                                   Characteristics      UOM       VALUE  
0                                Number of persons   Number  18759800.0  
1   Low 95% confidence interval, number of persons   Numbe

In [30]:
print(df['Characteristics'].unique())


StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 37, Finished, Available, Finished, False)

['Number of persons' 'Low 95% confidence interval, number of persons'
 'High 95% confidence interval, number of persons' 'Percent'
 'Low 95% confidence interval, percent'
 'High 95% confidence interval, percent'
 'Statistically different from previous reference period'
 'Statistically different from the Canada (excluding territories) rate'
 'Statistically different from the rest of Canada']


In [31]:
df = df[df['Characteristics'].isin(['Number of persons', 'Percent'])]
print(df.shape)

StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 38, Finished, Available, Finished, False)

(83908, 8)


In [32]:
print(df['GEO'].unique())

StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 39, Finished, Available, Finished, False)

['Canada (excluding territories)' 'Newfoundland and Labrador'
 'Prince Edward Island' 'Nova Scotia' 'New Brunswick' 'Quebec' 'Ontario'
 'Manitoba' 'Saskatchewan' 'Alberta' 'British Columbia']


In [33]:
print(df['Sex'].unique())

StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 40, Finished, Available, Finished, False)

['Both sexes' 'Males' 'Females']


In [34]:
print(df['REF_DATE'].unique())

StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 41, Finished, Available, Finished, False)

[2015 2016 2017 2018 2019 2020 2021 2022]


In [35]:
print(df['VALUE'].isnull().sum())
print(df[df['VALUE'].isnull()].head(10))

StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 42, Finished, Available, Finished, False)

18815
     REF_DATE                             GEO       Age group         Sex  \
531      2015  Canada (excluding territories)  12 to 17 years  Both sexes   
534      2015  Canada (excluding territories)  12 to 17 years  Both sexes   
573      2015  Canada (excluding territories)  12 to 17 years  Both sexes   
576      2015  Canada (excluding territories)  12 to 17 years  Both sexes   
579      2015  Canada (excluding territories)  12 to 17 years  Both sexes   
582      2015  Canada (excluding territories)  12 to 17 years  Both sexes   
658      2015  Canada (excluding territories)  12 to 17 years       Males   
661      2015  Canada (excluding territories)  12 to 17 years       Males   
676      2015  Canada (excluding territories)  12 to 17 years       Males   
679      2015  Canada (excluding territories)  12 to 17 years       Males   

                                            Indicators    Characteristics  \
531  Chronic obstructive pulmonary disease (COPD; 3...  Number of per

In [36]:
df = df.rename(columns={
    'REF_DATE': 'year',
    'GEO': 'province',
    'Age group': 'age_group',
    'Sex': 'sex',
    'Indicators': 'indicator',
    'Characteristics': 'characteristic',
    'UOM': 'unit_of_measure',
    'VALUE': 'value'
})
print(df.columns.tolist())

StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 43, Finished, Available, Finished, False)

['year', 'province', 'age_group', 'sex', 'indicator', 'characteristic', 'unit_of_measure', 'value']


In [37]:
indicator_type_map = {
    'Perceived health, very good or excellent': 'positive',
    'Perceived mental health, very good or excellent': 'positive',
    'Self-reported physical activity, 150 minutes per week, adult (18 years and over)': 'positive',
    'Self-reported physical activity, average 60 minutes per day, youth (12 to 17 years old)': 'positive',
    'Breast milk feeding initiation': 'positive',
    'Exclusive breastfeeding, at least 6 months': 'positive',
    'Fruit and vegetable consumption, 5 times or more per day': 'positive',
    'Sense of belonging to local community, somewhat strong or very strong': 'positive',
    'Life satisfaction, satisfied or very satisfied': 'positive',
    'Has a regular healthcare provider': 'positive',
    'Contact with a medical doctor in the past 12 months': 'positive',
    'Influenza immunization in the past 12 months': 'positive',
    'Perceived health, fair or poor': 'negative',
    'Perceived mental health, fair or poor': 'negative',
    'Perceived life stress, most days quite a bit or extremely stressful': 'negative',
    'Body mass index, adjusted self-reported, adult (18 years and over), overweight': 'negative',
    'Body mass index, adjusted self-reported, adult (18 years and over), obese': 'negative',
    'Body mass index, self-reported, youth (12 to 17 years old), overweight or obese': 'negative',
    'Arthritis (15 years and over)': 'negative',
    'Diabetes': 'negative',
    'Asthma': 'negative',
    'Chronic obstructive pulmonary disease (COPD; 35 years and over)': 'negative',
    'High blood pressure': 'negative',
    'Mood disorder': 'negative',
    'Current smoker, daily or occasional': 'negative',
    'Current smoker, daily': 'negative',
    'Heavy drinking': 'negative',
    'Cannabis use, past 12 months': 'negative',
    'Cannabis frequency of use in the past 12 months, daily or almost daily': 'negative',
    'Anxiety disorder': 'negative',
    'Ever used e-cigarette or vaping device': 'negative',
    'Used e-cigarette or vaping device, past 30 days': 'negative' 
}

df['type'] = df['indicator'].map(indicator_type_map)

StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 44, Finished, Available, Finished, False)

In [38]:
spark.createDataFrame(df).write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("silver_health_indicators")

StatementMeta(, c61044fe-7529-4be1-add0-02c0641ac6f1, 45, Finished, Available, Finished, False)